# Example 9.5.1 / 1: Read Semicolon-Separated and Headerless Files

Companion notebook for **9.5.1 CSV Files** in the AMPL lab Documentation, section "Read Semicolon-Separated and Headerless Files". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Read Semicolon-Separated and Headerless Files

Many European spreadsheet exports use `;` as the separator. `sep=semicolon` reads them, and
`header=false` reads a file without a header row, taking columns in declaration order:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;
param profit {PROD};
param market {PROD} >= 0;

table SemiColon IN "amplcsv" "products_semicolon.csv" "sep=semicolon":
    PROD <- [product], profit, market;
table NoHeader IN "amplcsv" "products_noheader.csv" "header=false":
    [product], profit, market;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
with open("products_semicolon.csv", "w") as f:
    f.write("product;profit;market\nbands;25;6000\ncoils;30;4000\nplate;29;3500\n")
with open("products_noheader.csv", "w") as f:
    f.write("bands,26,6100\ncoils,31,4100\nplate,30,3600\n")

In [5]:
# run.py
ampl.eval("load amplcsv.dll;")
ampl.read_table("SemiColon")
print(ampl.param["market"].to_dict())
ampl.read_table("NoHeader")
print(ampl.param["market"].to_dict())

{'bands': 6000, 'coils': 4000, 'plate': 3500}
{'bands': 6100, 'coils': 4100, 'plate': 3600}


### Pure AMPL: model.run

The second read replaces the values from the first, because both tables fill the same
parameters. Without `header=false`, `amplcsv` would take the first data row as the header.

Pure AMPL, with the same `model.mod` and files:

In [6]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [7]:
# data.py
with open("products_semicolon.csv", "w") as f:
    f.write("product;profit;market\nbands;25;6000\ncoils;30;4000\nplate;29;3500\n")
with open("products_noheader.csv", "w") as f:
    f.write("bands,26,6100\ncoils,31,4100\nplate,30,3600\n")

In [8]:
%%ampl_eval
# model.run
load amplcsv.dll;
read table SemiColon;
display market;
read table NoHeader;
display market;

market [*] :=
bands  6000
coils  4000
plate  3500
;

market [*] :=
bands  6100
coils  4100
plate  3600
;

